In [ ]:
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Bachelor thesis presentation

## Regularizing complex-valued thresholds in numerical integration of loop integrals

$$
    I = \int \frac{d^4{k}}{{(2\pi)}^4} \frac{i}{D_1 \, D_2 \, D_3}
$$
with
$$
    D_i = {(\mathbf{k}-\mathbf{q}_i)}^2-{(m_i - i\epsilon)}^2
$$

## **Problem:** Not suited for numerical integration

## **Solution:** Use CFF representation



$$
    I =  \int d^3{k} \; \mathcal{I}_{CFF} \\
    \mathcal{I}_{CFF} = \frac{1}{{(4\pi)}^3} \frac{1}{E_1E_2E_3} \left(
    \frac{1}{\eta_{21}^{++}\eta_{31}^{++}}
    + \frac{1}{\eta_{12}^{++}\eta_{13}^{++}}
    + \frac{1}{\eta_{12}^{++}\eta_{32}^{++}}
    + \frac{1}{\eta_{21}^{++}\eta_{23}^{++}}
    + \frac{1}{\eta_{13}^{++}\eta_{23}^{++}}
    + \frac{1}{\eta_{31}^{++}\eta_{32}^{++}}
    \right)
$$

## choosing coordinates that simplify $\eta$

$$
\eta_{ij}^{++} =  \sqrt{{(\vec{k'} - \vec{q})}^2 + m_i^2} + \sqrt{{(\vec{k'} + \vec{q})}^2 + m_j^2} + 2q^0
$$



$$
\begin{align*}
    \vec{k'} & = \vec{k} - \frac{\vec{q}_i}{2}(\vec{q}_i + \vec{q}_j) \\
    \mathbf{q}  & = \frac{1}{2}\left(\mathbf{q}_i-\mathbf{q}_j\right) 
\end{align*}
$$

## Let us understand the geometry of $\eta_{ij}^{++}$ better

In [ ]:
def e_surf(k, q, m1, m2):
    return (
        np.sqrt(np.sum((k - q[1:]) ** 2, axis=-1) + m1**2)
        + np.sqrt(np.sum((k + q[1:]) ** 2, axis=-1) + m2**2)
        - 2 * q[0]
    )

In [ ]:
res = 100
x = y = z = np.linspace(-1.2, 1.2, res)

ks = np.stack(np.meshgrid(x, y, z), axis = -1)

q = np.array([1,0,0,0.5])
m1 = 0.5
m2 = 0.5

vals = e_surf(ks, q, m1, m2)

grid = pv.ImageData()
grid.dimensions = np.array(vals.shape) # type: ignore
grid.origin = (x[0], y[0], z[0])
grid.spacing = (x[1] - x[0], y[1] - y[0], z[1] - z[0])

grid.point_data["vals"] = vals.flatten(order="F")
surf = grid.contour([0]) # type: ignore

plotter = pv.Plotter() # type: ignore
plotter.add_mesh(surf, color="cyan", opacity=0.5, smooth_shading=True)

arrow = pv.Arrow(start=(0,0,0), direction=q[1:]/q[0], scale=0.3)
plotter.add_mesh(arrow, color="red")
plotter.show_grid()
plotter.show(interactive=False)

/home/cedricsigrist/miniconda3/envs/triangler/lib/python3.12/site-packages/trame/ui/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


Widget(value='<iframe src="http://localhost:46477/index.html?ui=P_0x7f737c35f350_0&reconnect=auto" class="pyvi…

## Threshold subtraction

We want to subtract the singularities.
How can this notion be analytically continued?

**Idea:** Parameterize spherically and analytically continue in the radius.

Lets look at the following:

## $$\eta_{ij}^{++} (\vec{k}' = \hat{k} \; k) \qquad k \in \mathbb{C}$$

In [ ]:
def plot_complex_plane(xs, ys, ax = None):
    """Plot a complex→complex function using HSV color encoding for phase and magnitude.
    xs is a 2D grid (from np.meshgrid) of complex-plane x-values, ys is the complex output.
    NaN or inf values in ys are handled gracefully and shown as transparent.
    """
    
    if ax is None:
        ax = plt.gca()

    # Mask invalid data
    valid_mask = np.isfinite(ys)
    if not np.any(valid_mask):
        raise ValueError("All ys values are NaN or inf — nothing to plot.")

    # Compute phase and magnitude safely
    phase = np.angle(np.where(valid_mask, ys, 0))
    mag = np.abs(np.where(valid_mask, ys, 0))
    max_mag = np.nanmax(mag)
    mag = mag / max_mag if max_mag != 0 else mag

    # HSV mapping
    hue = (phase + np.pi) / (2 * np.pi)
    value = mag

    # HSV → RGB
    rgb = plt.cm.hsv(hue) # type: ignore
    rgb[..., :3] *= value[..., None]

    # Add transparency for invalid values
    alpha = np.where(valid_mask, 1.0, 0.0)
    rgb[..., -1] = alpha

    # Compute plotting extents (robust to NaNs)
    x_real = np.real(xs)
    y_imag = np.imag(xs)
    x_min, x_max = np.nanmin(x_real), np.nanmax(x_real)
    y_min, y_max = np.nanmin(y_imag), np.nanmax(y_imag)

    # Plot
    plt.imshow(
        rgb,
        origin="lower",
        extent=[x_min, x_max, y_min, y_max], # type: ignore
        interpolation="nearest",
        aspect="equal",  # maintain correct aspect ratio
    )

def plot_complex(xs, ys):
    """
    Plot a real -> complex function
    """
    plt.plot(xs, ys.real, label="re")
    plt.plot(xs, ys.imag, label="im")

In [ ]:
## ~ 7 second runtime

x = np.linspace(-2,2, 300)
y = np.linspace(-2,2, 300)
X, Y = np.meshgrid(x, y)

xs = X + 1j*Y

k_hat = np.array([0,0,1])
fig, ax = plt.subplots()

# Animation function
def update(frame):
    m1 = frame
    m2 = frame
    y = 1 / e_surf(xs[..., None] * k_hat, q, m1, m2)
    plot_complex_plane(xs, y, ax=ax)
    ax.set_xlabel("Re(k)")
    ax.set_ylabel("Im(k)")
    ax.set_title(f"mi = mj = {m1:.2f}")



m_values = np.linspace(0.5, 1, 30) - 0.01j # <=== show what happens with imaginary part

ani = FuncAnimation(fig, update, frames=m_values, interval=200) # type: ignore
ani.save("anim.gif", writer="pillow")
display(HTML(ani.to_jshtml()))
plt.close()


## We need to solve the following:

$$
\eta_{ij}^{++} (\vec{k}' = \hat{k} \; k) = 0 \qquad k \in \mathbb{C}
$$

### This can be done by some annoying algebraic manipulation, shown in the thesis to obtain a quadratic equation.
**This approach only works for the triangle integral!**

$$
    \alpha k^2 + \beta k + \gamma = 0
$$
$$
\begin{align*}
    \alpha & = 1-{(\hat{\mathbf{k}}\cdot\vec{v})}^2                                                                             \\
    \beta  & = 2(\hat{\mathbf{k}}\cdot\vec{k'_0})-2(\hat{\mathbf{k}}\cdot\vec{v})(\vec{k'_0}\cdot\vec{v})-2\Delta(\hat{\mathbf{k}}\cdot\vec{v}) \\
    \gamma & = \vec{k'_0}^2 - {(\vec{k'_0}\cdot\vec{v})}^2-2\Delta(\vec{k'_0}\cdot\vec{v}) - \mathbf{q}^2+\braket{m^2} - \Delta^2
\end{align*}
$$
with
$$
    \vec{v} = \frac{\vec{q}}{q^0}
    \qquad
    \braket{m^2} = \frac{m_i^2 + m_j^2}{2}
    \qquad
    \Delta = \frac{m_i^2 - m_j^2}{4q^0}
$$

### We are now able to identify the complex-valued zeros of $\eta_{ij}^{++}$ along any line.